# S&P 500 Current Constituents: 15-Year Time Series

This notebook is designed to be self-contained and portable.

It does not depend on a prebuilt returns CSV. Instead it:

1. detects the project root relative to where the notebook is run
2. fetches the **current** S&P 500 constituents table from Wikipedia
3. extracts current tickers, company names, sectors, and industries
4. downloads up to 15 years of daily adjusted price history with `yfinance`
5. writes two versions of the resulting time series:
   - a `wide` matrix for calculations
   - a `long` table for filtering, joins, and charting
6. writes metadata describing the run and missing tickers

## Important interpretation note

This notebook works with **current S&P 500 constituents viewed backward in time**.

That means:

- the ticker universe comes from the current Wikipedia S&P 500 list
- the 15-year history is downloaded for those current companies
- this is **not** a point-in-time historical reconstruction of index membership

In other words, this notebook answers:

> "For the companies that are in the S&P 500 today, what do their last 15 years of adjusted daily prices look like?"

It does **not** answer:

> "What were the exact constituents of the S&P 500 on each historical date?"

## Files written by this notebook

- `data/manual/notebook/sp500_current_constituents.csv`
- `data/manual/notebook/sp500_timeseries_15y_wide.csv`
- `data/manual/notebook/sp500_timeseries_15y_long.csv`
- `data/manual/notebook/sp500_timeseries_15y_meta.json`


## Cell 1: Environment and output paths

This cell prepares the notebook environment.

What it does:

- imports the libraries used later
- tries to detect the project root from the current working directory or one of its parents
- defines the output folder and output file paths
- defines the Wikipedia URL used as the source of the current S&P 500 constituents table
- creates the output directory if it does not exist

Why this matters:

- it avoids hard-coded machine-specific paths like `/Users/...`
- it allows the notebook to work from another machine after cloning the repo
- it makes all outputs predictable and repeatable inside the repo structure


In [ ]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup

cwd = Path.cwd().resolve()

def looks_like_project_root(path: Path) -> bool:
    return (path / 'data').exists() and (path / 'notebooks').exists()

candidates = [cwd, *cwd.parents]
ROOT = next((path for path in candidates if looks_like_project_root(path)), cwd)

if ROOT == cwd and not looks_like_project_root(ROOT):
    print(f'Warning: project root not detected. Using current working directory: {ROOT}')

OUT_DIR = ROOT / 'data' / 'manual' / 'notebook'

CONSTITUENTS_OUT = OUT_DIR / 'sp500_current_constituents.csv'
WIDE_OUT = OUT_DIR / 'sp500_timeseries_15y_wide.csv'
LONG_OUT = OUT_DIR / 'sp500_timeseries_15y_long.csv'
META_OUT = OUT_DIR / 'sp500_timeseries_15y_meta.json'

WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('ROOT =', ROOT)
print('OUT_DIR =', OUT_DIR)


## Cell 2: Download and parse the current S&P 500 constituents table

This cell builds the current company universe.

What it does:

- requests the Wikipedia page for the S&P 500 constituents
- parses the HTML table with `BeautifulSoup`
- extracts four core fields:
  - `ticker`
  - `name`
  - `sector`
  - `industry`
- normalizes tickers like `BRK.B` to `BRK-B` for Yahoo compatibility
- writes the current constituent list to `sp500_current_constituents.csv`

Why save this intermediate CSV:

- it gives you a traceable snapshot of the exact current universe used in the run
- it lets you inspect the ticker and sector mapping before the historical download
- it makes debugging missing symbols much easier


In [ ]:
def normalize_ticker(symbol: str) -> str:
    return symbol.strip().replace('.', '-')


resp = requests.get(
    WIKI_URL,
    headers={'User-Agent': 'Mozilla/5.0'},
    timeout=30,
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
table = soup.select_one('#constituents')
if table is None:
    raise ValueError('No se encontró la tabla #constituents en Wikipedia')

rows = []
for tr in table.select('tbody tr'):
    tds = [td.get_text(' ', strip=True) for td in tr.select('td')]
    if len(tds) < 4:
        continue

    rows.append({
        'ticker': normalize_ticker(tds[0]),
        'name': tds[1],
        'sector': tds[2],
        'industry': tds[3],
    })

constituents_df = pd.DataFrame(rows)
constituents_df.to_csv(CONSTITUENTS_OUT, index=False)
constituents_df.head()


## Cell 3: Extract the ticker universe

This cell reduces the constituent table to the list of symbols that Yahoo Finance will use.

What it does:

- takes the `ticker` column from the constituent table
- removes nulls
- forces string type
- trims whitespace
- removes duplicates
- converts the result to a plain Python list

Why this is separated into its own cell:

- it makes it easy to inspect the ticker universe before the expensive historical download
- it gives a clean place to patch or exclude symbols if Yahoo fails on specific tickers


In [ ]:
tickers = (
    constituents_df['ticker']
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

len(tickers), tickers[:10]


## Cell 4: Download up to 15 years of daily adjusted price history

This is the raw market-data download step.

Parameters used:

- `period='15y'`
- `interval='1d'`
- `auto_adjust=True`
- `group_by='ticker'`

What comes back:

- a `pandas.DataFrame`
- usually with **MultiIndex columns** when many tickers are downloaded at once
- grouped first by ticker, then by field like `Open`, `High`, `Low`, `Close`, `Volume`

This is not the final table yet. It is the raw wide market download that still needs to be simplified.


In [ ]:
history = yf.download(
    tickers=tickers,
    period='15y',
    interval='1d',
    auto_adjust=True,
    group_by='ticker',
    threads=True,
    progress=False,
)

history.shape


## Cell 5: Transform the raw Yahoo download into `wide` format

This cell extracts a clean adjusted-close matrix.

Goal:

- keep only the `Close` series for each ticker
- build one table where:
  - each row is a date
  - each column is a ticker

Example conceptual shape:

```text
date         AAPL   MSFT   NVDA
2011-04-20   ...    ...    ...
2011-04-21   ...    ...    ...
```

Why `wide` matters:

- it is convenient for matrix-like calculations
- it is good for rolling windows, correlations, covariance, return matrices, and vectorized transformations

The function also collects `missing_tickers`, which is useful when Yahoo returns nothing for some symbols.


In [ ]:
def extract_close_wide(frame: pd.DataFrame, tickers: list[str]) -> tuple[pd.DataFrame, list[str]]:
    if frame.empty:
        return pd.DataFrame(), tickers

    close_map: dict[str, pd.Series] = {}
    missing: list[str] = []

    if isinstance(frame.columns, pd.MultiIndex):
        available = set(frame.columns.get_level_values(0))
        for ticker in tickers:
            if ticker not in available:
                missing.append(ticker)
                continue

            sub = frame[ticker]
            if 'Close' not in sub:
                missing.append(ticker)
                continue

            series = sub['Close'].copy()
            if series.dropna().empty:
                missing.append(ticker)
                continue

            close_map[ticker] = series
    else:
        if 'Close' in frame:
            close_map[tickers[0]] = frame['Close'].copy()
        else:
            missing.extend(tickers)

    wide = pd.DataFrame(close_map)
    wide.index = pd.to_datetime(wide.index)
    wide.index.name = 'date'
    wide = wide.sort_index()
    return wide, missing


close_wide, missing_tickers = extract_close_wide(history, tickers)
close_wide.shape, len(missing_tickers)


## Cell 6: Transform `wide` into `long` and write outputs

This cell creates the second standard shape of the same time series.

What `long` means:

- each row is one `(date, ticker)` observation
- instead of many ticker columns, ticker becomes a categorical column

Example conceptual shape:

```text
date         ticker   adj_close
2011-04-20   AAPL     ...
2011-04-20   MSFT     ...
2011-04-20   NVDA     ...
```

Why `long` matters:

- it is easier to filter by ticker, sector, or industry
- it is more natural for plotting libraries and data joins
- it is often the better shape for analytics notebooks and tidy-data workflows

This same cell also:

- merges in metadata from the constituents table
- creates the `meta` dictionary
- writes `wide.csv`, `long.csv`, and `meta.json`


In [ ]:
close_long = (
    close_wide.reset_index()
    .melt(id_vars='date', var_name='ticker', value_name='adj_close')
    .dropna(subset=['adj_close'])
    .merge(constituents_df[['ticker', 'name', 'sector', 'industry']], on='ticker', how='left')
    .sort_values(['ticker', 'date'])
    .reset_index(drop=True)
)

meta = {
    'generatedAt': datetime.now(timezone.utc).isoformat(),
    'source': 'Current S&P 500 constituents from Wikipedia',
    'sourceUrl': WIKI_URL,
    'period': '15y',
    'interval': '1d',
    'autoAdjust': True,
    'tickersRequested': len(tickers),
    'tickersReturned': int(close_wide.shape[1]),
    'missingTickers': missing_tickers,
    'wideShape': [int(close_wide.shape[0]), int(close_wide.shape[1])],
    'longShape': [int(close_long.shape[0]), int(close_long.shape[1])],
    'note': 'This dataset uses current S&P 500 constituents and looks backward up to 15 years. It is not a point-in-time historical membership reconstruction.',
}

close_wide.to_csv(WIDE_OUT)
close_long.to_csv(LONG_OUT, index=False)
META_OUT.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

meta


## Cell 7: Print a human-readable summary

The final cell is just a small reporting step.

It prints:

- where each output file was saved
- how many tickers were missing from Yahoo
- a small preview of the missing list

This is useful because notebook runs can succeed while still silently missing a few symbols. This final print makes that visible immediately.


In [ ]:
print('Saved:')
print('-', CONSTITUENTS_OUT)
print('-', WIDE_OUT)
print('-', LONG_OUT)
print('-', META_OUT)

print('\nMissing tickers:', len(missing_tickers))
print(missing_tickers[:25])
